In [1]:
# Please edit the path according to your PC environment
from google.colab import drive
drive.mount('/content/drive')

!ls '/content/drive/MyDrive/counterfactual_pitch_sequence_analysis/'
%cd "/content/drive/MyDrive/counterfactual_pitch_sequence_analysis/"

Mounted at /content/drive
data  dataset  models  program	results
/content/drive/MyDrive/counterfactual_pitch_sequence_analysis


In [2]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import tensorflow as tf
from tensorflow.keras import layers, models

In [3]:
# Define transformer model
class PaddingMaskLayer(layers.Layer):
    def call(self, inputs):
        is_padding = tf.reduce_all(tf.equal(inputs, 0.0), axis=-1)
        valid_mask = tf.logical_not(is_padding)
        return valid_mask

class PositionalEmbedding(layers.Layer):
    def __init__(self, seq_len, d_model, **kwargs):
        super().__init__(**kwargs)
        self.seq_len = seq_len
        self.d_model = d_model
        self.pos_emb = layers.Embedding(
            input_dim=seq_len,
            output_dim=d_model)

    def call(self, x):
        positions = tf.range(start=0, limit=self.seq_len, delta=1)
        pos = self.pos_emb(positions)
        return x + pos

    def get_config(self):
        config = super().get_config()
        config.update({
            "seq_len": self.seq_len,
            "d_model": self.d_model})
        return config

class TransformerEncoderBlock(layers.Layer):
    def __init__(
        self,
        d_model,
        num_heads,
        ff_dim,
        dropout_rate=0.2,
        **kwargs):
        super().__init__(**kwargs)

        self.d_model = d_model
        self.num_heads = num_heads
        self.ff_dim = ff_dim
        self.dropout_rate = dropout_rate

        self.attn = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=d_model // num_heads,
            dropout=dropout_rate)

        self.dropout1 = layers.Dropout(dropout_rate)
        self.norm1 = layers.LayerNormalization()

        self.ffn = models.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dropout(dropout_rate),
            layers.Dense(d_model)])

        self.dropout2 = layers.Dropout(dropout_rate)
        self.norm2 = layers.LayerNormalization()

    def call(self, inputs, training=False):
        x, valid_mask = inputs
        attn_mask = tf.cast(valid_mask[:, tf.newaxis, :], tf.bool)
        attn_output = self.attn(
            x, x,
            attention_mask=attn_mask,
            training=training)
        attn_output = self.dropout1(attn_output, training=training)

        x = self.norm1(x + attn_output)

        ffn_output = self.ffn(x, training=training)
        ffn_output = self.dropout2(ffn_output, training=training)

        x = self.norm2(x + ffn_output)

        return x

    def get_config(self):
        config = super().get_config()
        config.update({
            "d_model": self.d_model,
            "num_heads": self.num_heads,
            "ff_dim": self.ff_dim,
            "dropout_rate": self.dropout_rate})
        return config

class MaskedMeanPooling(layers.Layer):
    def call(self, inputs):
        x, valid_mask = inputs
        mask = tf.cast(valid_mask, tf.float32)
        mask = mask[:, :, tf.newaxis]
        x = x * mask
        sum_x = tf.reduce_sum(x, axis=1)
        count = tf.reduce_sum(mask, axis=1)
        return sum_x / tf.maximum(count, 1.0)

# Define helper functions
def plate_x_norm_to_ft(x_norm):
    return (2.0 * x_norm - 1.0) * PLATE_HALF_WIDTH_FT

def transform_sequence_standardizer(X, mean, std):
    X_scaled = X.copy().astype(np.float32)
    flat = X_scaled.reshape(-1, X_scaled.shape[-1])
    non_padding_mask = ~(np.all(flat == 0, axis=1))
    flat[non_padding_mask] = (flat[non_padding_mask] - mean) / std
    return X_scaled

def transform_tabular_standardizer(C, mean, std):
    return ((C.astype(np.float32) - mean) / std).astype(np.float32)

def predict_model_output(X_seq, C_context):
    X_seq = np.asarray(X_seq, dtype=np.float32)
    C_context = np.asarray(C_context, dtype=np.float32)

    X_scaled = transform_sequence_standardizer(X_seq, x_mean, x_std)
    C_scaled = transform_tabular_standardizer(C_context, c_mean, c_std)

    preds = model.predict(
        {"pitch_sequence": X_scaled,
         "context_features": C_scaled},
        verbose=0)

    return preds.ravel()

def add_plate_z_norm(df):
    df = df.copy()
    df["plate_z_norm"] = ((df["plate_z"] - df["sz_bot"]) / (df["sz_top"] - df["sz_bot"]))
    return df

def normalize_x_for_batter_view(df):
    df = df.copy()
    x_cols = ["pfx_x", "plate_x"]
    left_mask = df["stand"] == "L"
    for col in x_cols:
        if col in df.columns:
            df.loc[left_mask, col] = -df.loc[left_mask, col]
    return df

def load_pitch_type_table(source_file):
    path = PITCH_TYPE_ROOT / source_file
    df = pd.read_csv(path)
    features = [
        "pitch_type",
        "pfx_x","pfx_z",
        "effective_speed",
        "release_spin_rate",
        "spin_axis"]
    for col in features[1:]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.dropna(subset=features)
    return df

def load_raw_source_file(source_file):
    path = RAW_DATA_ROOT / source_file
    df = pd.read_csv(path)

    numeric_cols = [
        "game_pk", "at_bat_number",
        "pitcher", "batter",
        "pitch_number", "strikes",
        "plate_x", "plate_z",
        "sz_top", "sz_bot",
        "effective_speed", "release_spin_rate",
        "spin_axis", "pfx_x", "pfx_z"]

    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df = add_plate_z_norm(df)
    df = normalize_x_for_batter_view(df)

    return df

def find_original_target_and_last_pitch(meta_row):
    source_file = meta_row["source_file"]
    raw_df = load_raw_source_file(source_file)

    cond = (
        (raw_df["game_pk"] == meta_row["game_pk"]) &
        (raw_df["at_bat_number"] == meta_row["at_bat_number"]) &
        (raw_df["pitcher"] == meta_row["pitcher"]) &
        (raw_df["batter"] == meta_row["batter"]))

    g = raw_df.loc[cond].copy()
    g = g.sort_values("pitch_number")
    target_pitch = g.iloc[-2]
    last_pitch = g.iloc[-1]

    return target_pitch, last_pitch, None

def make_counterfactual_rows(i, meta_row, pitch_type_df):
    original_seq = X_test[i].copy()
    original_context = C_test[i].copy()

    target_pitch, last_pitch, reason = find_original_target_and_last_pitch(meta_row)

    sz_top = target_pitch.get("sz_top", np.nan)
    sz_bot = target_pitch.get("sz_bot", np.nan)

    rows = []

    original_output = predict_model_output(
        original_seq.reshape(1, MAX_PITCHES, len(FEATURE_COLUMNS)),
        original_context.reshape(1, -1))[0]

    original_target_feature = dict(zip(FEATURE_COLUMNS, original_seq[-2]))
    original_last_feature = dict(zip(FEATURE_COLUMNS, original_seq[-1]))

    rows.append({
        "row_type": "original_target",
        "region": "original_target",
        "pitch_type": target_pitch.get("pitch_type", np.nan),
        "grid_x_index": np.nan,
        "grid_z_index": np.nan,
        "effective_speed": original_target_feature["effective_speed"],
        "release_spin_rate": original_target_feature["release_spin_rate"],
        "spin_axis": original_target_feature["spin_axis"],
        "pfx_x": original_target_feature["pfx_x"],
        "pfx_z": original_target_feature["pfx_z"],
        "plate_x": original_target_feature["plate_x"],
        "plate_x_norm": np.nan,
        "plate_z_norm": original_target_feature["plate_z_norm"],
        "plate_z": target_pitch.get("plate_z", np.nan),
        "sz_bot": target_pitch.get("sz_bot", np.nan),
        "sz_top": target_pitch.get("sz_top", np.nan),
        "model_output": original_output})

    rows.append({
        "row_type": "original_last",
        "region": "original_last",
        "pitch_type": last_pitch.get("pitch_type", np.nan),
        "grid_x_index": np.nan,
        "grid_z_index": np.nan,
        "effective_speed": original_last_feature["effective_speed"],
        "release_spin_rate": original_last_feature["release_spin_rate"],
        "spin_axis": original_last_feature["spin_axis"],
        "pfx_x": original_last_feature["pfx_x"],
        "pfx_z": original_last_feature["pfx_z"],
        "plate_x": original_last_feature["plate_x"],
        "plate_x_norm": np.nan,
        "plate_z_norm": original_last_feature["plate_z_norm"],
        "plate_z": last_pitch.get("plate_z", np.nan),
        "sz_bot": last_pitch.get("sz_bot", np.nan),
        "sz_top": last_pitch.get("sz_top", np.nan),
        "model_output": original_output})

    candidate_seqs = []
    candidate_meta = []
    stand = target_pitch.get("stand", None)
    x_sign = -1.0 if stand == "L" else 1.0
    for _, pt in pitch_type_df.iterrows():
        pitch_type = pt["pitch_type"]

        for region, grid in REGIONS.items():
            x_norm_values = grid["x_norm_values"]
            z_norm_values = grid["z_norm_values"]

            for ix, plate_x_norm in enumerate(x_norm_values):
                for iz, plate_z_norm in enumerate(z_norm_values):

                    seq_cf = original_seq.copy()
                    plate_x = plate_x_norm_to_ft(plate_x_norm)
                    plate_z_actual = sz_bot + plate_z_norm * (sz_top - sz_bot)

                    cf_pfx_x = pt["pfx_x"] * x_sign
                    seq_cf[-2, FEATURE_COLUMNS.index("effective_speed")] = pt["effective_speed"]
                    seq_cf[-2, FEATURE_COLUMNS.index("release_spin_rate")] = pt["release_spin_rate"]
                    seq_cf[-2, FEATURE_COLUMNS.index("spin_axis")] = pt["spin_axis"]
                    seq_cf[-2, FEATURE_COLUMNS.index("pfx_x")] = cf_pfx_x
                    seq_cf[-2, FEATURE_COLUMNS.index("pfx_z")] = pt["pfx_z"]
                    seq_cf[-2, FEATURE_COLUMNS.index("plate_x")] = plate_x
                    seq_cf[-2, FEATURE_COLUMNS.index("plate_z_norm")] = plate_z_norm

                    candidate_seqs.append(seq_cf)
                    candidate_meta.append({
                        "row_type": "counterfactual",
                        "region": region,
                        "pitch_type": pitch_type,
                        "grid_x_index": ix,
                        "grid_z_index": iz,
                        "effective_speed": pt["effective_speed"],
                        "release_spin_rate": pt["release_spin_rate"],
                        "spin_axis": pt["spin_axis"],
                        "pfx_x": cf_pfx_x,
                        "pfx_z": pt["pfx_z"],
                        "plate_x": plate_x,
                        "plate_x_norm": plate_x_norm,
                        "plate_z_norm": plate_z_norm,
                        "plate_z": plate_z_actual,
                        "sz_bot": sz_bot,
                        "sz_top": sz_top})

    if candidate_seqs:
        candidate_seqs = np.stack(candidate_seqs).astype(np.float32)
        candidate_contexts = np.repeat(
            original_context.reshape(1, -1),
            len(candidate_seqs),
            axis=0).astype(np.float32)

        outputs = predict_model_output(candidate_seqs, candidate_contexts)
        for m, output in zip(candidate_meta, outputs):
            m["model_output"] = output
            rows.append(m)

    result_df = pd.DataFrame(rows)

    return result_df, None

In [4]:
DATASET_PATH = Path("./dataset/set_up/dataset_set_up.npz")
META_TEST_PATH = Path("./dataset/set_up/meta_test_set_up.csv")

MODEL_PATH = Path("./models/transformer.keras")
STANDARDIZER_PATH = Path("./models/transformer_train_info.npz")

RAW_DATA_ROOT = Path("./data/pitch_sequence/2025")
PITCH_TYPE_ROOT = Path("./data/pitch_type/2025")

RESULT_DIR = Path("./results/analysis2")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

MAX_PITCHES = 6
FEATURE_COLUMNS = [
    "effective_speed",
    "release_spin_rate",
    "spin_axis",
    "pfx_x",
    "pfx_z",
    "plate_x",
    "plate_z_norm"]

PLATE_HALF_WIDTH_FT = 8.5 / 12.0

# Define the grid for the ball zone
INSIDE_GRID = np.array([0, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 1.0])
LOW_OUTSIDE_GRID = np.array([-0.10, -0.20])
HIGH_OUTSIDE_GRID = np.array([1.10, 1.20])

# 8 regions:
# left, right, down, up,
# down_left, down_right, up_left, up_right
REGIONS = {
    "left": {
        "x_norm_values": LOW_OUTSIDE_GRID,
        "z_norm_values": INSIDE_GRID},
    "right": {
        "x_norm_values": HIGH_OUTSIDE_GRID,
        "z_norm_values": INSIDE_GRID},
    "down": {
        "x_norm_values": INSIDE_GRID,
        "z_norm_values": LOW_OUTSIDE_GRID},
    "up": {
        "x_norm_values": INSIDE_GRID,
        "z_norm_values": HIGH_OUTSIDE_GRID},
    "down_left": {
        "x_norm_values": LOW_OUTSIDE_GRID,
        "z_norm_values": LOW_OUTSIDE_GRID},
    "down_right": {
        "x_norm_values": HIGH_OUTSIDE_GRID,
        "z_norm_values": LOW_OUTSIDE_GRID},
    "up_left": {
        "x_norm_values": LOW_OUTSIDE_GRID,
        "z_norm_values": HIGH_OUTSIDE_GRID},
    "up_right": {
        "x_norm_values": HIGH_OUTSIDE_GRID,
        "z_norm_values": HIGH_OUTSIDE_GRID}}

data = np.load(DATASET_PATH, allow_pickle=True)

X_test = data["X_test"]
C_test = data["C_test"]
y_test = data["y_test"] if "y_test" in data.files else None

meta_test = pd.read_csv(META_TEST_PATH)

model = tf.keras.models.load_model(
    MODEL_PATH,
    custom_objects={
        "PaddingMaskLayer": PaddingMaskLayer,
        "PositionalEmbedding": PositionalEmbedding,
        "TransformerEncoderBlock": TransformerEncoderBlock,
        "MaskedMeanPooling": MaskedMeanPooling})

std_data = np.load(STANDARDIZER_PATH, allow_pickle=True)

x_mean = std_data["x_mean"].astype(np.float32)
x_std = std_data["x_std"].astype(np.float32)
c_mean = std_data["c_mean"].astype(np.float32)
c_std = std_data["c_std"].astype(np.float32)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:424: UserWarning: `build()` was called on layer 'positional_embedding', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:424: UserWarning: `build()` was called on layer 'transformer_encoder_block_1', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:424: UserWarning: `build()` was called on layer 'transformer_enco

In [5]:
# Running counterfactual analysis
for i, meta_row in tqdm(meta_test.iterrows(), total=len(meta_test)):
    source_file = meta_row["source_file"]
    player_name = Path(source_file).stem
    pitch_type_df = load_pitch_type_table(source_file)
    result_df, reason = make_counterfactual_rows(i, meta_row, pitch_type_df)
    original_target_row = result_df[result_df["row_type"] == "original_target"].iloc[0]
    original_last_row = result_df[result_df["row_type"] == "original_last"].iloc[0]
    cf_df = result_df[result_df["row_type"] == "counterfactual"].copy()

    # save the results
    year = meta_row["year"]
    player_dir_name = f"{year}_{player_name}"
    player_dir = RESULT_DIR / player_dir_name
    player_dir.mkdir(parents=True, exist_ok=True)
    out_name = f"{i:06d}.csv"
    out_path = player_dir / out_name
    result_df.to_csv(out_path, index=False)

print("Finish all counterfactual analysis")

# (Subsequent analyses are performed in MATLAB)

  0%|          | 0/5680 [00:00<?, ?it/s]

Finish all counterfactual analysis


In [6]:
from google.colab import runtime
runtime.unassign()